In [ ]:
import math
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d, gaussian_filter
from scipy.signal import find_peaks, savgol_filter

from data_analysis.io import open_lapd
from Jan2024_Isat import init_read, get_Isat_ratio, setup_2subplots, get_IV, find_IV_tndx, find_Te, get_Te_all
from data_analysis.plasma.langmuir import EEDF, derivative, find_Vp


%matplotlib qt
plt.rcParams.update({'font.size': 16})

In [ ]:
ifn = r"D:\data\LAPD\JAN2024_diverging_B\07_2cmXYline_M1P24_M3P27_2024-01-27_15.11.19.hdf5"

with open_lapd(ifn).session() as sess:
    sess.info()

pr_area = {
    'M3': np.array([1,1,1,1.15]) * 0.2*0.4,
    'M1': np.array([1,1,1.14,1]) * 0.1*0.4
} # [UL, UR, LL, LR] ; unit:cm^2

In [5]:
# Reads and initializes data.
adc, digi_dict, pos_array, xpos, ypos, zpos, npos, nshot, int_arr, int_tarr = init_read(ifn)

SIS Crate activated:
3302 board 1
Channel 1 -- active: TRUE -- description: p27_M3_UL
Channel 2 -- active: TRUE -- description: p27_M3_UR
Channel 3 -- active: TRUE -- description: p27_M3_LL
Channel 4 -- active: TRUE -- description: p27_M3_LR
Channel 5 -- active: FALSE -- description:  
Channel 6 -- active: FALSE -- description:  
Channel 7 -- active: TRUE -- description: Vsweep_supply
Channel 8 -- active: FALSE -- description:  
3302 board 2
Channel 1 -- active: TRUE -- description: p24_M1_UL
Channel 2 -- active: TRUE -- description: p24_M1_UR
Channel 3 -- active: TRUE -- description: p24_M1_LL
Channel 4 -- active: TRUE -- description: p24_M1_LR
Channel 5 -- active: FALSE -- description:  
Channel 6 -- active: FALSE -- description:  
Channel 7 -- active: FALSE -- description:  
Channel 8 -- active: FALSE -- description:  
2 probes found.
Langmuir Marvin_3  at port  27
Only one motion list found: Xline-Y0-dx2-Nx31
Number of positions: 31
Number of shots per position: 11
This probe moves

c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\utils\hdfreadcontrols.py:315: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  data = np.empty(shape, dtype=dtype)
c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\utils\hdfreadcontrols.py:315: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  data = np.empty(shape, dtype=dtype)


In [ ]:
# IV sweep data
with open_lapd(ifn).session() as sess:
    tarr, Vsweep_arr, IsweepL_dic, IsweepR_dic = get_IV(sess, adc, npos, nshot, pr_area, cal_fac = [1, 1.414])

In [ ]:
# Isat data and ratio
with open_lapd(ifn).session() as sess:
    tarr, Isat_UL_dic, Isat_UR_dic, I_ratio = get_Isat_ratio(sess, adc, digi_dict, npos, nshot, pr_area, R=[7.8,9.7], bg_tind=60000) #R=[7.8,9.7]

In [ ]:
I_ratio[2][:,30000]

In [9]:
start_ls = [1000, 12000, 22000, 32000, 42000]
stop_ls = [52000, 13000, 23000, 33000, 43000]
pos_ind = 15
pos = list(pos_array[1][pos_ind*nshot])

plt.figure()
plt.title(f'Isat ratio at x={round(pos[0],1)}cm, y={round(pos[1],1)}cm')
plt.xlabel('t (ms)')
plt.ylabel('Isat ratio')

plt.plot(tarr[start_ls[0]:stop_ls[0]]*1e3, I_ratio[1][pos_ind,start_ls[0]:stop_ls[0]], label='P27')
plt.plot(tarr[start_ls[0]:stop_ls[0]]*1e3, I_ratio[2][pos_ind,start_ls[0]:stop_ls[0]], label='P24')

plt.ylim([0, 1])

plt.legend()
plt.grid(True)
plt.tight_layout()

for i in range(1, 5):
    plt.plot(tarr[start_ls[i]:stop_ls[i]]*1e3, I_ratio[1][pos_ind,start_ls[i]:stop_ls[i]], color='black')
    plt.plot(tarr[start_ls[i]:stop_ls[i]]*1e3, I_ratio[2][pos_ind,start_ls[i]:stop_ls[i]], color='black')

In [45]:
colors = plt.cm.rainbow(np.linspace(0, 1, 5))
ax = setup_2subplots(ylabel='Isat ratio')

for i in range(1,5):
    I_ratio_avg = np.mean(I_ratio[1][:,start_ls[i]:stop_ls[i]], axis=-1)
    ax[0].scatter(xpos, I_ratio_avg, marker='o', color=colors[i], label='t'+str(i))

    I_ratio_avg = np.mean(I_ratio[2][:,start_ls[i]:stop_ls[i]], axis=-1)
    ax[1].scatter(xpos, I_ratio_avg, marker='^', color=colors[i], label='t'+str(i))
        

ax[1].legend(loc='lower center', ncol=4, handletextpad=0.1, columnspacing=0.1)

ax[0].set_ylim([0,1])
ax[1].set_ylim([0,1])
plt.xlim([-25, 25])
plt.tight_layout()

In [30]:
colors = plt.cm.rainbow(np.linspace(0, 1, 5))
ax = setup_2subplots(ylabel="I_sat (A/cm^2)")

for i in range(1,5):
    sig = np.mean(Isat_UL_dic[1][:,:,start_ls[i]:stop_ls[i]], axis=(1,2))
    ax[0].scatter(xpos, sig, marker='o', color=colors[i], label='t'+str(i))

    sig = np.mean(Isat_UL_dic[2][:,:,start_ls[i]:stop_ls[i]], axis=(1,2))
    ax[1].scatter(xpos, sig, marker='^', color=colors[i], label='t'+str(i))

ax[1].legend(loc='upper center', bbox_to_anchor=(0.08, 1.2), handletextpad=0.1, columnspacing=0.1)
ax[0].set_ylim([0, 2.2])
ax[1].set_ylim([0, 2.2])
plt.tight_layout()

In [ ]:
with open_lapd(ifn).session() as sess:
    message_array, status_array, timestamp_array = sess.datarun_sequence()

In [ ]:
print(message_array[0:10])